## Transforming the Archivo Eltit - Rosenfeld digital collection to RDF

Created in July-September 2026 for the Pontificia Universidad Católica de Chile by Gustavo Candela

This dataset represents the descriptive metadata from the Archivo Eltit-Rosenfeld, a documentary collection created by the artist Lotty Rosenfeld and the writer Diamela Eltit in the late 1980s and early 1990s to preserve the testimonies and memory of the women’s movement for the right to vote in Chile. 

- Data format: metadata available as Dublin Core by means of an OAI-PMH server
- Data source: https://archivospatrimoniales.uc.cl/handle/123456789/31557

### Preparation

Import the libraries required to explore the summary of each record included in the dataset

In [100]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import FOAF, RDF, DCTERMS, VOID, DC, SKOS, OWL
import datetime
import xml.etree.ElementTree as ET
import unicodedata

### Transformation to RDF

*Note: The variable domain could be updated to the domain of the organisation (e.g., https://archivospatrimoniales.uc.cl/).

In [101]:
domain = 'https://example.org/'
domainLanguage = domain + 'language/'

# folder to store the RDF outputs
folder_name = 'chile'
file_name = 'chile'

First, we instantiate all the namespaces that we will use when defining the RDF data

In [102]:
g = Graph()
g.bind("foaf", FOAF)
g.bind("rdf", RDF)
g.bind("dcterms", DCTERMS)
g.bind("dc", DC)
g.bind("void", VOID)
g.bind("skos", SKOS)
g.bind("owl", OWL)

schema = Namespace("https://schema.org/")
g.bind("schema", schema)

viaf = Namespace("https://viaf.org/viaf/")
g.bind("viaf", viaf)

wd = Namespace("http://www.wikidata.org/entity/")
g.bind("wd", wd)

### We define the dataset Archivo Eltit - Rosenfeld

In [103]:
eltit = URIRef(domain + "dataset/eltit")
g.add((eltit, RDF.type, schema.Dataset))
g.add((eltit, schema.url, URIRef("https://archivospatrimoniales.uc.cl/handle/123456789/31557")))
g.add((eltit, schema.description, Literal("El Archivo Eltit-Rosenfeld es un fondo documental creado por la artista Lotty Rosenfeld y la escritora Diamela Eltit a fines de la década del 80 y principios de la del 90, para el rescate de los testimonios y la memoria del movimiento de mujeres por el derecho a voto en Chile")))
g.add((eltit, schema.name, Literal("Archivo Eltit - Rosenfeld")))
g.add((eltit, DC.title, Literal("Archivo Eltit - Rosenfeld")))
g.add((eltit, schema.license, URIRef('https://creativecommons.org/publicdomain/zero/1.0/')))
g.add((eltit, schema.address, Literal("Biblioteca de Humanidades, 3er piso - Vicuña Mackenna 4860, Macul")))
g.add((eltit, schema.email, Literal("archivosuc@uc.cl")))

now = datetime.datetime.now()
g.add((eltit, schema.dateCreated, Literal(str(now)[:10])))

<Graph identifier=N2b0dafef1d7f4919b1872ffb75b3ff6e (<class 'rdflib.graph.Graph'>)>

### We define the University

In [104]:
pucc = URIRef(domain + "organisation/pucc")
g.add((pucc, RDF.type, schema.Organization))
g.add((pucc, schema.url, URIRef("https://www.uc.cl/")))
g.add((pucc, schema.logo, URIRef("https://www.uc.cl/site/templates/dist/images/logo-uc-wh.svg")))
g.add((pucc, schema.name, Literal("Pontificia Universidad Católica de Chile")))
g.add((pucc, DC.title, Literal("Pontificia Universidad Católica de Chile")))

<Graph identifier=N2b0dafef1d7f4919b1872ffb75b3ff6e (<class 'rdflib.graph.Graph'>)>

### We open the XML records retrieved from the OAI server:
https://archivospatrimoniales.uc.cl/oai/request?verb=ListRecords&metadataPrefix=oai_dc&set=com_123456789_31557

In [105]:
ns = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "oai_dc": "http://www.openarchives.org/OAI/2.0/oai_dc/",
    "dc": "http://purl.org/dc/elements/1.1/"
}

tree = ET.parse("../datos/archivospatrimoniales_set_31557.xml")
root = tree.getroot()

### We define a function to create uris from text

In [106]:
def createUri(text):

    cleanText = text;
    if "/" in cleanText:
        cleanText = cleanText[0:cleanText.index("/")-1]
    
    cleanText = cleanText.lower().strip().replace("’", "")\
        .replace(".", "").replace("(", "").replace(")", "")\
        .replace(",", "").replace("‘", "").replace(" ", "-")

    cleanText = ''.join(
    c for c in unicodedata.normalize('NFD', cleanText)
    if unicodedata.category(c) != 'Mn')

    # duplicated names
    if "Vergara" in text:
        cleanText = "vergara-paredes-hernan"
    
    return cleanText

### We extract each record and transform it to RDF using schema.org as main ontology

In [107]:
for record in root.findall(".//oai:record", ns):
    dc = record.find(".//oai_dc:dc", ns)

    datos = {}

    for elem in dc:
        tag = elem.tag.split("}")[-1]
        datos.setdefault(tag, []).append(elem.text)

    #print("Nuevo registro")

    identifier = ""
    uri = ""
    title = ""
    description = "" 
    typeTxt = ""
    externalUrl = ""
    publishedDate = ""
    dateCreated = ""
    encodingFormat = ""
    creator = ""
    
    for campo, valores in datos.items():
        #if campo == "title":
        print(campo, "->", valores)

        if campo == "identifier":
            uri = valores[0]
            if len(valores) >1 :
               identifier = valores[0].strip() 
               uri  = valores[1].strip()
        elif campo == "format":
            encodingFormat = valores[0].strip()
            if len(valores) > 1 :
                encodingFormat = valores[1].strip()
        elif campo == "title":
            title = valores[0].strip()
        elif campo == "description":
            description = valores[0].strip()
            if len(valores) > 1:
                externalUrl = valores[1].strip()
        elif campo == "type":
            typeTxt = valores[0].strip()
        elif campo == "creator":
            creator = valores[0].strip()
        elif campo == "date":
            publishedDate = valores[0].strip()
            if len(valores) > 2:
                dateCreated = valores[len(valores)-1].strip()
        else:
            pass

    classtype = 'https://schema.org/CreativeWork'
    classtypeImage = 'https://schema.org/ImageObject'
    classtypeVideo = 'https://schema.org/VideoObject'
    classtypeText = 'https://schema.org/Text'

    
    record = URIRef(uri.strip())
    g.add((record, RDF.type, URIRef(classtype)))
    if typeTxt == "Fotografía":
        g.add((record, RDF.type, URIRef(classtypeImage)))
    elif typeTxt == "Video":
        g.add((record, RDF.type, URIRef(classtypeVideo)))
    elif typeTxt == "Manuscrito" or typeTxt == "Documento":
        g.add((record, RDF.type, URIRef(classtypeText)))
    else:
        pass
    
    g.add((record, schema.sourceOrganization, pucc))
    g.add((record, schema.isPartOf, eltit))
    if identifier != "":
        g.add((record, schema.identifier, Literal(identifier)))
    g.add((record, schema.datePublished, Literal(publishedDate)))
    if dateCreated != "":
        g.add((record, schema.dateCreated, Literal(dateCreated)))
    g.add((record, schema.name, Literal(title)))
    g.add((record, schema.encodingFormat, Literal(encodingFormat)))
    g.add((record, schema.additionalType, Literal(typeTxt)))
    if externalUrl != "":
        g.add((record, schema.url, URIRef(externalUrl)))
    g.add((record, schema.abstract, Literal(description)))
    g.add((record, schema.license, URIRef('https://creativecommons.org/publicdomain/zero/1.0/')))

    ## Author
    if creator != "":
        author = URIRef(domain + 'author/' + createUri(creator))
                    
        g.add((record, schema.author, author))
        g.add((author, RDF.type, schema.Person))
        g.add((author, RDF.type, FOAF.Person))
        g.add((author, SKOS.prefLabel, Literal(creator)))
        g.add((author, schema.name, Literal(creator)))
        g.add((author, FOAF.name, Literal(creator)))         

title -> ['Collage de fotografías de María de la Cruz']
description -> ['Collage de dos fotos en blanco y negro. Una de ellas muestra, parte de un desfile de Partido Feminista en Chile y la otra a María de la Cruz sentada junto a un hombre.']
date -> ['2020-09-10T02:59:52Z', '2020-09-10T02:59:52Z']
type -> ['Fotografía']
identifier -> ['01_05_14', 'https://archivospatrimoniales.uc.cl/handle/123456789/31660']
relation -> ['Archivo Eltit Rosenfeld', 'Fotografías']
format -> ['1 fotografía en blanco y negro.', 'application/pdf']
title -> ['María de la Cruz junto a su hijo en Buenos Aires']
description -> ['Fotografía apaisada en blanco y negro. Aparece María de la Cruz junto a su hijo y un par de caballos en Buenos Aires.']
date -> ['2020-09-10T02:59:58Z', '2020-09-10T02:59:58Z']
type -> ['Fotografía']
identifier -> ['01_05_04', 'https://archivospatrimoniales.uc.cl/handle/123456789/31672']
relation -> ['Archivo Eltit Rosenfeld', 'Fotografías']
format -> ['1 fotografía en blanco y negro.',

### We store the graph in the form of a ttl file

In [108]:
g.serialize(destination="../datos/output/"+folder_name+"/dataset_"+file_name+".ttl")

<Graph identifier=N2b0dafef1d7f4919b1872ffb75b3ff6e (<class 'rdflib.graph.Graph'>)>